# 🤖 Stage 4 — Handling Imbalance, Model Building & Evaluation
### 🏦 Loan Default Predictor — Home Credit Dataset
---
**Apply SMOTE, train Logistic Regression / Random Forest / XGBoost, and compare results.**

> **Sections 6, 7 & 8**

```
Progress: ████░  Stage 4 of 5
```

← [Stage 3](stage_03_preprocessing_and_features.ipynb)  |  [Stage 5](stage_05_shap_and_export.ipynb) →

---
### 📋 What you will do in this stage:
- Apply **SMOTE** to fix the class imbalance in training data
- Train three models: **Logistic Regression**, **Random Forest**, **XGBoost**
- Understand the difference between **bagging** and **boosting**
- Compare models using **AUC-ROC**, **Precision-Recall**, and **F1-score**
- Build **ROC curves**, **Precision-Recall curves**, and **Confusion Matrices**
- Analyze **feature importance** from XGBoost

⏱️ *Estimated time: 60–90 minutes*

---
> ⚠️ **Prerequisite:** This notebook depends on **Stage 3 (preprocessed `X_train_scaled`, `X_test_scaled`, `y_train`, `y_test`)**.  
> Run the previous stage(s) first, **or** run the cell below to reload saved objects.


### ⚙️ Reload Cell
Run this if you are starting this notebook fresh (without Stage 3 in memory).
It re-runs the full preprocessing pipeline so you can jump straight to modeling.


In [ ]:
# ── RELOAD: Full preprocessing pipeline — run if starting fresh ──
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12

df = pd.read_csv("data/raw/application_train.csv")

FEATURES = [
    "EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3",
    "AMT_INCOME_TOTAL","AMT_CREDIT","AMT_ANNUITY","AMT_GOODS_PRICE",
    "DAYS_BIRTH","DAYS_EMPLOYED","CODE_GENDER","NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE","NAME_FAMILY_STATUS","NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE","FLAG_OWN_CAR","FLAG_OWN_REALTY",
    "CNT_CHILDREN","CNT_FAM_MEMBERS","REGION_RATING_CLIENT",
    "REG_CITY_NOT_WORK_CITY","DEF_30_CNT_SOCIAL_CIRCLE"
]
TARGET = "TARGET"
X = df[FEATURES].copy(); y = df[TARGET].copy()

# Fix anomaly
X["DAYS_EMPLOYED"] = X["DAYS_EMPLOYED"].replace(365243, 0)

# Impute
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
for c in num_cols: X[c] = X[c].fillna(X[c].median())
for c in cat_cols: X[c] = X[c].fillna("Unknown")

# Encode
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Engineer features
X["CREDIT_INCOME_RATIO"]  = X["AMT_CREDIT"]  / (X["AMT_INCOME_TOTAL"] + 1)
X["ANNUITY_INCOME_RATIO"] = X["AMT_ANNUITY"] / (X["AMT_INCOME_TOTAL"] + 1)

# Split & scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
feature_names  = X_train.columns.tolist()

print(f"✅ Preprocessing complete: {X_train_scaled.shape[1]} features")
print(f"   Train: {len(X_train):,}  |  Test: {len(X_test):,}")

---
## Section 6 — Handling Class Imbalance

Remember: only ~8% of applicants defaulted. If we ignore this, the model will be biased  
toward always predicting "no default" — which is safe but useless for a bank!

### Two main strategies:

| Strategy | How it works |
|----------|-------------|
| **Class weights** | Tell the model: "mistakes on the minority class cost more" |
| **SMOTE** | Synthetically generate new minority class examples |

We'll use **SMOTE** (Synthetic Minority Over-sampling Technique) on the training set only.

> 💡 **What SMOTE does:**  
> It doesn't just copy existing defaulters — it creates **new synthetic examples**  
> by interpolating between existing minority samples. This gives the model more  
> varied examples to learn from.

> ⚠️ **Important:** We only apply SMOTE to the **training set**.  
> The test set must stay imbalanced to reflect real-world conditions.


In [ ]:
# Check class distribution before SMOTE
print("Before SMOTE:")
print(f"  Non-default (0): {(y_train == 0).sum():,}")
print(f"  Default    (1): {(y_train == 1).sum():,}")
print(f"  Ratio: 1 defaulter per {int((y_train==0).sum()/(y_train==1).sum())} repaid")

In [ ]:
# Apply SMOTE — this may take a minute on the full dataset
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("
After SMOTE:")
print(f"  Non-default (0): {(y_train_res == 0).sum():,}")
print(f"  Default    (1): {(y_train_res == 1).sum():,}")
print(f"  New training size: {len(X_train_res):,} samples")

In [ ]:
# Visualize the rebalancing
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before
axes[0].bar(["Repaid (0)", "Defaulted (1)"],
            [(y_train==0).sum(), (y_train==1).sum()],
            color=["steelblue", "tomato"], edgecolor="black")
axes[0].set_title("Before SMOTE", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")

# After
axes[1].bar(["Repaid (0)", "Defaulted (1)"],
            [(y_train_res==0).sum(), (y_train_res==1).sum()],
            color=["steelblue", "tomato"], edgecolor="black")
axes[1].set_title("After SMOTE (Balanced)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Count")

plt.suptitle("Effect of SMOTE on Class Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 7 — Model Building

We'll train **three different models** and compare them:

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **Logistic Regression** | Simple, fast, interpretable | Assumes linear relationships |
| **Random Forest** | Handles non-linearity, robust to noise | Slower, less interpretable |
| **XGBoost** | Usually best performance, handles missing data | Many hyperparameters to tune |

> 💡 **Why start with Logistic Regression?**  
> It's the "baseline" — if a simple model works well, we don't need complexity.  
> We also use it to check that our preprocessing pipeline is working correctly.


### 7.1 — Evaluation Metrics Setup

We'll evaluate every model using the same metrics:

- **AUC-ROC**: Area Under the ROC Curve — 0.5 = random, 1.0 = perfect. Best overall metric.
- **Precision**: Of all predicted defaults, how many were actual defaults?
- **Recall**: Of all actual defaults, how many did we catch?
- **F1-score**: Harmonic mean of precision and recall

> 💡 **For a bank, Recall is critical.**  
> Missing a real defaulter (False Negative) is expensive — the bank loses the loan amount.  
> Flagging a good payer as defaulter (False Positive) is less costly — the bank just turns them away.


In [ ]:
# Utility function — evaluate any model consistently
def evaluate_model(model_name, y_true, y_pred_proba, threshold=0.5):
    """Print and return key metrics for a classifier."""
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    auc  = roc_auc_score(y_true, y_pred_proba)
    ap   = average_precision_score(y_true, y_pred_proba)
    
    print(f"{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(f"  Avg Prec : {ap:.4f}")
    print(f"
{classification_report(y_true, y_pred, target_names=['Repaid', 'Defaulted'])}")
    
    return {"model": model_name, "auc": auc, "avg_precision": ap}

results = []  # We'll collect all model results here

### 7.2 — Logistic Regression (Baseline)


In [ ]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_res, y_train_res)

print("✅ Logistic Regression trained.")

In [ ]:
# Evaluate on test set
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]  # Probability of default
lr_result = evaluate_model("Logistic Regression", y_test, lr_proba)
results.append(lr_result)

### 7.3 — Random Forest

A Random Forest builds many decision trees on random subsets of data,  
then averages their predictions. This reduces overfitting and improves generalization.

> 💡 **Analogy — The wisdom of crowds:**  
> Instead of asking one expert, you ask 100 different experts and take the majority vote.  
> Each expert (tree) sees slightly different data, so they make different mistakes.  
> Combined, they cancel each other's errors out.


In [ ]:
# Train Random Forest
# n_estimators = number of trees (more = better but slower)
# class_weight handles imbalance as a backup strategy
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1  # Use all available CPU cores
)
rf.fit(X_train_res, y_train_res)

print("✅ Random Forest trained.")

In [ ]:
# Evaluate on test set
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]
rf_result = evaluate_model("Random Forest", y_test, rf_proba)
results.append(rf_result)

### 7.4 — XGBoost

XGBoost (Extreme Gradient Boosting) builds trees **sequentially**, where each tree  
learns from the errors of the previous one. This makes it very powerful.

> 💡 **Boosting vs. Bagging:**  
> Random Forest: trees trained **independently** (parallel)  
> XGBoost: trees trained **sequentially** — each corrects previous mistakes  
> Think of Bagging as asking 100 people at once, Boosting as learning from your mistakes.


In [ ]:
# Calculate the ratio of negatives to positives for scale_pos_weight
# This helps XGBoost handle imbalance internally
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_ratio = neg_count / pos_count

print(f"scale_pos_weight = {scale_ratio:.2f}  (weights minority class {scale_ratio:.0f}x higher)")

In [ ]:
# Train XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_ratio,  # Handle imbalance
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_res, y_train_res, verbose=False)

print("✅ XGBoost trained.")

In [ ]:
# Evaluate on test set
xgb_proba = xgb.predict_proba(X_test_scaled)[:, 1]
xgb_result = evaluate_model("XGBoost", y_test, xgb_proba)
results.append(xgb_result)

---
## Section 8 — Model Comparison & Deep Evaluation

Now let's compare all three models visually and select the best one.


In [ ]:
# Summary table
results_df = pd.DataFrame(results).set_index("model")
print("Model Comparison Summary:")
print(results_df.round(4).to_string())

In [ ]:
# Bar chart — AUC-ROC comparison
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["steelblue", "seagreen", "darkorange"]
bars = ax.bar(results_df.index, results_df["auc"], color=colors, edgecolor="black")
ax.set_ylim(0.5, 0.85)
ax.set_title("AUC-ROC Score Comparison", fontsize=14, fontweight="bold")
ax.set_ylabel("AUC-ROC")
ax.axhline(0.5, color="red", linestyle="--", label="Random baseline (0.5)")
ax.legend()

for bar, val in zip(bars, results_df["auc"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{val:.4f}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

### 8.1 — ROC Curves

The ROC curve plots **True Positive Rate (Recall)** vs. **False Positive Rate** at every threshold.  
A curve that hugs the top-left corner = better model.  
The diagonal line = random guessing (AUC = 0.5).


In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(9, 7))

for name, proba, color in [
    ("Logistic Regression", lr_proba, "steelblue"),
    ("Random Forest",       rf_proba, "seagreen"),
    ("XGBoost",             xgb_proba, "darkorange")
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", linewidth=2, color=color)

# Diagonal = random guessing
ax.plot([0, 1], [0, 1], "k--", label="Random (AUC=0.5)")

ax.set_title("ROC Curves — Model Comparison", fontsize=14, fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("reports/figures/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.2 — Confusion Matrix (Best Model)

A confusion matrix shows us exactly where the model makes mistakes:

|  | Predicted: Repaid | Predicted: Default |
|--|------------------|--------------------|
| **Actual: Repaid**  | ✅ True Negative  | ❌ False Positive |
| **Actual: Default** | ❌ False Negative | ✅ True Positive  |

For a bank, **False Negatives** (missed defaults) are the most costly.


In [ ]:
# Use XGBoost — typically the best performer
best_proba = xgb_proba
best_pred  = (best_proba >= 0.5).astype(int)

cm = confusion_matrix(y_test, best_pred)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Repaid", "Defaulted"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — XGBoost", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("reports/figures/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"
False Negatives (missed defaults): {cm[1][0]:,}")
print(f"False Positives (wrongly flagged) : {cm[0][1]:,}")

### 8.3 — Precision-Recall Curve

For imbalanced datasets, the Precision-Recall curve is often **more informative** than ROC.  
It focuses specifically on the minority class (defaulters).

- High **Precision** = when we say "default", we're usually right
- High **Recall** = we catch most actual defaults

The bank must choose the right **threshold** based on its risk appetite.


In [ ]:
# Precision-Recall curve
fig, ax = plt.subplots(figsize=(9, 6))

for name, proba, color in [
    ("Logistic Regression", lr_proba, "steelblue"),
    ("Random Forest",       rf_proba, "seagreen"),
    ("XGBoost",             xgb_proba, "darkorange")
]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(rec, prec, label=f"{name} (AP={ap:.4f})", linewidth=2, color=color)

# Baseline (random classifier on imbalanced data)
baseline = y_test.mean()
ax.axhline(baseline, color="black", linestyle="--", label=f"Baseline (AP={baseline:.3f})")

ax.set_title("Precision-Recall Curves", fontsize=14, fontweight="bold")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("reports/figures/pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.4 — Feature Importance

Which features does the model rely on most?  
XGBoost provides built-in feature importance scores.


In [ ]:
# Get feature importances from XGBoost
feature_names = X_train.columns.tolist()
importances = pd.Series(xgb.feature_importances_, index=feature_names)
top_20 = importances.sort_values(ascending=False).head(20)

print("Top 20 Most Important Features:")
print(top_20.round(4).to_string())

In [ ]:
# Bar chart of feature importance
fig, ax = plt.subplots(figsize=(11, 7))
colors_fi = plt.cm.RdYlGn_r(np.linspace(0, 1, 20))
top_20.sort_values().plot(kind="barh", ax=ax, color=colors_fi, edgecolor="black")
ax.set_title("Top 20 Feature Importances — XGBoost", fontsize=14, fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig("reports/figures/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

---
## ✅ Stage 4 Complete!

Great work! Here is a summary of what you accomplished:

- Apply **SMOTE** to fix the class imbalance in training data
- Train three models: **Logistic Regression**, **Random Forest**, **XGBoost**
- Understand the difference between **bagging** and **boosting**
- Compare models using **AUC-ROC**, **Precision-Recall**, and **F1-score**
- Build **ROC curves**, **Precision-Recall curves**, and **Confusion Matrices**
- Analyze **feature importance** from XGBoost

⏱️ *Estimated time: 60–90 minutes*

---
### ➡️ Next: 🧠 Stage 5 — Interpretability, Save & Export
**Explain predictions with SHAP, save the final model, and export results for Tableau.**

Open **`stage_05_shap_and_export.ipynb`** to continue.
